In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- Define the full file paths you provided ---
PATH_FINAL_DATASET = r"E:\Project\datasets\data\final_crop_yield_dataset.csv"
PATH_CROP_YIELD_DATASET = r"E:\Project\datasets\data\_crop+yield+prediction_data_crop_yield.csv"

# --- Part 1: Analyze the Merged "Master" Dataset ---

print(f"Loading master file: {PATH_FINAL_DATASET}...")
try:
    # We load ONLY the master file, using the full path
    df_final = pd.read_csv(PATH_FINAL_DATASET, low_memory=False)
except FileNotFoundError:
    print(f"Error: File not found at '{PATH_FINAL_DATASET}'")
    print("Please double-check the path and try again. Exiting.")
    # In a real script, we'd exit()
    df_final = None 
except Exception as e:
    print(f"An error occurred loading {PATH_FINAL_DATASET}: {e}")
    df_final = None

if df_final is not None:
    print("Preprocessing df_final...")
    # 1. Handle Missing Data
    df_final.dropna(inplace=True)

    # 2. Handle Date Column
    try:
        if 'Date' in df_final.columns:
            df_final['Date'] = pd.to_datetime(df_final['Date'], errors='coerce')
            df_final.dropna(subset=['Date'], inplace=True) # Drop rows where date failed
            df_final['Month'] = df_final['Date'].dt.month
            df_final['Day'] = df_final['Date'].dt.day
            df_final = df_final.drop(columns=['Date'])
    except Exception as e:
        print(f"Warning processing Date: {e}")

    # 3. Encode Categorical Columns
    le = LabelEncoder()
    categorical_cols = df_final.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        df_final[col] = le.fit_transform(df_final[col])

    print("Calculating correlation matrix for df_final...")
    # Calculate the correlation matrix
    corr_matrix_final = df_final.corr()

    # --- Create and show a heatmap WITH VALUES ---
    print("Displaying heatmap for the final_crop_yield_dataset (with values)...")
    print("NOTE: This may be very dense and hard to read.")
    
    plt.figure(figsize=(24, 22))
    sns.heatmap(
        corr_matrix_final, 
        annot=True,                 # <-- THIS IS THE CHANGE: Show values
        cmap='coolwarm', 
        fmt='.2f',                  # Format values to 2 decimal places
        annot_kws={"size": 8}       # Set the font size for the values
    )
    plt.title('Correlation Heatmap for final_crop_yield_dataset.csv (with values)', fontsize=20)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show() # Show the plot

    # --- Extracting Interesting Correlations from the big matrix ---
    print("\n--- High & Low Correlations in final_crop_yield_dataset.csv ---")

    # Unstack the matrix to a Series
    corr_series = corr_matrix_final.unstack()
    corr_series = corr_series.drop_duplicates()
    corr_series = corr_series[corr_series != 1] # drop self-correlations

    # Sort the series
    sorted_corr = corr_series.sort_values(ascending=False)

    print("\n** Top 20 MOST POSITIVE Correlations **")
    print(sorted_corr.head(20))

    print("\n** Top 20 MOST NEGATIVE Correlations **")
    print(sorted_corr.tail(20))


# --- Part 2: Analyze the Separate Yield Dataset ---

print(f"\n\nLoading separate file: {PATH_CROP_YIELD_DATASET}...")
try:
    # We load the SECOND file, using the full path
    df_crop_yield = pd.read_csv(PATH_CROP_YIELD_DATASET)
except FileNotFoundError:
    print(f"Error: File not found at '{PATH_CROP_YIELD_DATASET}'")
    print("Please double-check the path.")
    df_crop_yield = None
except Exception as e:
    print(f"An error occurred loading {PATH_CROP_YIELD_DATASET}: {e}")
    df_crop_yield = None

if df_crop_yield is not None:
    # Preprocess: Encode 'Crop'
    le_crop = LabelEncoder()
    df_crop_yield['Crop'] = le_crop.fit_transform(df_crop_yield['Crop'])

    print("Calculating correlation matrix for df_crop_yield...")
    corr_matrix_yield = df_crop_yield.corr()

    print("\n--- Correlations with 'Yield' (from _crop+yield+prediction_data_crop_yield.csv) ---")
    # Get correlations with Yield, then sort
    yield_corr = corr_matrix_yield['Yield'].sort_values(ascending=False)
    print(yield_corr)

print("\nAnalysis complete.")